In [75]:
#Importing all the librraies

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                              roc_curve, roc_auc_score)

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize']=10,6

In [76]:
from os import error
df=pd.read_csv("/content/Titanic-Dataset.csv",encoding_errors='replace')
print("="*60)
print("1. DATA OVERVIEW")
print(f"Shape:{df.shape}")
print(df.dtypes)
print("\nFirst 5 Rows:")
print(df.head())

1. DATA OVERVIEW
Shape:(891, 12)
PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object

First 5 Rows:
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male

In [77]:
#2 MISSING VALUES/ DATA QUALITY
print("\n"+"="*60)
print("2. MISSING VALUES")
print("="*60)

missing=df.isnull().sum()
missing_pct=(missing/len(df)*100).round(2)
missing_df=pd.DataFrame({"Missing_count":missing,"Missing_pct":missing_pct})
print(missing_df[missing_df.Missing_count>0])


2. MISSING VALUES
          Missing_count  Missing_pct
Age                 177        19.87
Cabin               687        77.10
Embarked              2         0.22


In [78]:
#DESCRIPTIVE STATISTICS

print("\n" + "="*60)
print("3. DESCRIPTIVE STATISTICS")
print("="*60)

print(df.describe(include="all").T)

age_clean=df["Age"].dropna()

print(f"\nAge - mean: {np.mean(age_clean):.2f}, median: {np.median(age_clean):.2f}, "
      f"std: {np.std(age_clean):.2f}, skew: {stats.skew(age_clean):.2f}, "
      f"kurtosis: {stats.kurtosis(age_clean):.2f}")

fare_clean=df["Fare"].dropna()

print(f"\nFare - Mean :{np.mean(fare_clean):.2f},median: {np.median(fare_clean):.2f},"
      f"std:{np.std(fare_clean):.2f},skew:{stats.skew(fare_clean):.2f},")


3. DESCRIPTIVE STATISTICS
             count unique                  top freq       mean         std  \
PassengerId  891.0    NaN                  NaN  NaN      446.0  257.353842   
Survived     891.0    NaN                  NaN  NaN   0.383838    0.486592   
Pclass       891.0    NaN                  NaN  NaN   2.308642    0.836071   
Name           891    891  Dooley, Mr. Patrick    1        NaN         NaN   
Sex            891      2                 male  577        NaN         NaN   
Age          714.0    NaN                  NaN  NaN  29.699118   14.526497   
SibSp        891.0    NaN                  NaN  NaN   0.523008    1.102743   
Parch        891.0    NaN                  NaN  NaN   0.381594    0.806057   
Ticket         891    681               347082    7        NaN         NaN   
Fare         891.0    NaN                  NaN  NaN  32.204208   49.693429   
Cabin          204    147                   G6    4        NaN         NaN   
Embarked       889      3            

In [79]:
#DATA CLEANING/FEATURE ENGINEERING
df['Age']=df['Age'].fillna(df['Age'].median())
df["Embarked"]=df["Embarked"].fillna(df['Embarked'].mode())
df["Fare"]=df["Fare"].fillna(df["Fare"].median())
df['CabinKnown']=df['Cabin'].notnull().astype(int)
df['Familysize']=df['SibSp']+df['Parch'] +1
df['IsAlone']=(df['Familysize']==1).astype(int)
df["Title"]=df['Name'].str.extract(r',\s*([^\.]*)\.')
rare_titles=df['Title'].value_counts()[df['Title'].value_counts()<10].index
df['Title']=df['Title'].replace(rare_titles,'Rare')
df['AgeBin']=pd.cut(df['Age'],bins=[0,12,18,35,60,100],labels=['Child','Teen','YoungAdult','Adult','Senior'])
df['FareBin']=pd.qcut(df['Fare'],4,labels=['low','Mid','High','Very High'])

df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,CabinKnown,Familysize,IsAlone,Title,AgeBin,FareBin
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,0,2,0,Mr,YoungAdult,low
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,1,2,0,Mrs,Adult,Very High
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,0,1,1,Miss,YoungAdult,Mid
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,1,2,0,Mrs,YoungAdult,Very High
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,0,1,1,Mr,YoungAdult,Mid
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S,0,1,1,Rare,YoungAdult,Mid
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S,1,1,1,Miss,YoungAdult,High
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,28.0,1,2,W./C. 6607,23.4500,NaN,S,0,4,0,Miss,YoungAdult,High
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C,1,1,1,Mr,YoungAdult,High


In [80]:
#HYPOTHESIS TESTING

print('\n'+"="*60)
print("5. STATISTICAL TESTS")
print("="*60)


5. STATISTICAL TESTS


In [81]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)


# ============================================================
# VISUALIZATIONS
# ============================================================

print("\n" + "=" * 60)
print("6. Visualizations")
print("=" * 60)


# ------------------------------------------------------------
# 1. BASIC SURVIVAL VISUALIZATIONS
# ------------------------------------------------------------

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Survival count
sns.countplot(
    data=df,
    x='Survived',
    ax=axes[0, 0],
    palette='Set2'
)

axes[0, 0].set_title(
    "Survival Count (0 = Died, 1 = Survived)"
)


# Survival by Sex
sns.countplot(
    data=df,
    x='Sex',
    hue='Survived',
    ax=axes[0, 1],
    palette='Set2'
)

axes[0, 1].set_title("Survival by Sex")


# Survival by Passenger Class
sns.countplot(
    data=df,
    x='Pclass',
    hue='Survived',
    ax=axes[0, 2],
    palette='Set2'
)

axes[0, 2].set_title("Survival by Passenger Class")


# Age distribution
sns.histplot(
    data=df,
    x='Age',
    hue='Survived',
    kde=True,
    bins=30,
    ax=axes[1, 0],
    palette='Set2'
)

axes[1, 0].set_title("Age Distribution by Survival")


# Fare by Class
sns.boxplot(
    data=df,
    x='Pclass',
    y='Fare',
    ax=axes[1, 1],
    palette='Set3'
)

axes[1, 1].set_title("Fare by Class")
axes[1, 1].set_ylim(0, 300)


# Survival by Embarkation
sns.countplot(
    data=df,
    x='Embarked',
    hue='Survived',
    ax=axes[1, 2],
    palette='Set2'
)

axes[1, 2].set_title("Survival by Embarkation Port")


plt.tight_layout()
plt.show()


# ============================================================
# 2. CORRELATION HEATMAP
# ============================================================

fig, ax = plt.subplots(figsize=(9, 7))

numeric_cols = [
    'Survived',
    'Pclass',
    'Age',
    'SibSp',
    'Parch',
    'Fare',
    'Familysize',
    'CabinKnown',
    'IsAlone'
]

corr = df[numeric_cols].corr()

sns.heatmap(
    corr,
    annot=True,
    cmap='coolwarm',
    center=0,
    fmt='.2f',
    ax=ax
)

ax.set_title("Correlation Heatmap")

plt.tight_layout()
plt.show()


# ============================================================
# 3. SURVIVAL RATE BY TITLE AND FAMILY SIZE
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))


# Survival by Title
title_surv = (
    df.groupby("Title")['Survived']
    .mean()
    .sort_values(ascending=False)
)

title_surv.plot(
    kind='bar',
    ax=axes[0],
    color=sns.color_palette(
        'viridis',
        len(title_surv)
    )
)

axes[0].set_title("Survival Rate by Title")
axes[0].set_ylabel("Survival Rate")
axes[0].set_xlabel("Title")


# Survival by Family Size
fam_surv = (
    df.groupby("Familysize")['Survived']
    .mean()
)

fam_surv.plot(
    kind='bar',
    ax=axes[1],
    color=sns.color_palette(
        'magma',
        len(fam_surv)
    )
)

axes[1].set_title("Survival Rate by Family Size")
axes[1].set_ylabel("Survival Rate")
axes[1].set_xlabel("Family Size")


plt.tight_layout()
plt.show()


6. Visualizations


/tmp/ipykernel_1411/2319039255.py:27: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.countplot(
/tmp/ipykernel_1411/2319039255.py:78: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
